# Backend Floor Planner Workflow Test

Run this notebook before testing inside SketchUp. It calls the hosted backend over HTTP so the same backend logs used by SketchUp are exercised.

Coverage:
- backend health and route availability through the hosted API
- floor-plan discussion draft capture
- continued discussion with `temporary_floor_plan_draft`
- LLM-supported decoration JSON and floor-plan SVG plotting tools through `/agent/orchestrate`
- direct `/generate/floor-plan` validation
- artifact download for generated decoration JSON, SVG, and PNG preview

In [ ]:
import base64
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

REPO_ROOT = Path.cwd()
BACKEND_ROOT = REPO_ROOT / "backend"
BACKEND_URL = os.getenv("BACKEND_URL", "http://127.0.0.1:8000").rstrip("/")

NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "outputs" / "notebook-floor-plan"
NOTEBOOK_EXPORT_DIR = REPO_ROOT / "exports" / "notebook-floor-plan"
NOTEBOOK_POINTCLOUD_DIR = REPO_ROOT / "pointclouds" / "notebook-floor-plan"
for path in (NOTEBOOK_OUTPUT_DIR, NOTEBOOK_EXPORT_DIR, NOTEBOOK_POINTCLOUD_DIR):
    path.mkdir(parents=True, exist_ok=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is required because FloorPlanPlotTool owns SVG plotting."

class ApiResponse:
    def __init__(self, status_code: int, body: bytes):
        self.status_code = status_code
        self.content = body
        self.text = body.decode("utf-8", errors="replace")

    def json(self):
        return json.loads(self.text)


class HostedBackendClient:
    def __init__(self, base_url: str):
        self.base_url = base_url

    def get(self, path: str) -> ApiResponse:
        return self._request("GET", path)

    def post(self, path: str, json_body: dict) -> ApiResponse:
        return self._request("POST", path, json_body)

    def _request(self, method: str, path: str, json_body: dict | None = None) -> ApiResponse:
        body = None if json_body is None else json.dumps(json_body).encode("utf-8")
        headers = {} if body is None else {"Content-Type": "application/json"}
        request = urllib.request.Request(f"{self.base_url}{path}", data=body, headers=headers, method=method)
        try:
            with urllib.request.urlopen(request, timeout=180) as response:
                return ApiResponse(response.status, response.read())
        except urllib.error.HTTPError as exc:
            return ApiResponse(exc.code, exc.read())


client = HostedBackendClient(BACKEND_URL)
print("Backend URL:", BACKEND_URL)
print("Watch backend logs with: docker compose logs -f backend")
print("Backend root:", BACKEND_ROOT)
print("Notebook output dir:", NOTEBOOK_OUTPUT_DIR)


## 1. Backend Health And Routes

In [ ]:
health = client.get("/health")
assert health.status_code == 200
assert health.json() == {"status": "ok"}

openapi = client.get("/openapi.json")
assert openapi.status_code == 200, openapi.text
routes = set(openapi.json()["paths"].keys())
required_routes = {
    "/agent/orchestrate",
    "/generate/floor-plan",
    "/artifacts/download",
}
assert required_routes.issubset(routes), sorted(required_routes - routes)

{"health": health.json(), "routes_checked": sorted(required_routes)}


## 2. Shared Render Payload Helper

In [ ]:
def valid_render_payload(user_prompt: str) -> dict:
    return {
        "project_id": "notebook-floor-plan",
        "viewport_image_path": "notebook_view.png",
        "style": "modern interior",
        "user_prompt": user_prompt,
        "camera": {
            "position": [1.2, 3.4, 2.1],
            "direction": [0.0, 1.0, -0.2],
            "fov": 45,
        },
        "model": {
            "bounds": {"width": 8.2, "depth": 6.4, "height": 3.1},
            "materials": ["wood", "glass", "concrete"],
            "selected_entity_count": 0,
        },
        "render_options": {
            "preserve_geometry": True,
            "preserve_camera": True,
            "output_resolution": "1024x1024",
        },
    }


## 3. Discussion Draft Capture

This tests the backend discussion step that SketchUp uses before showing `Plot Floor Plan`.

In [ ]:
discussion_payload = valid_render_payload(
    "Floor plan with Living 12x14 and Kitchen 8x10 adjacent with a door"
)
discussion_response = client.post("/agent/orchestrate", json_body=discussion_payload)
assert discussion_response.status_code == 200, discussion_response.text
discussion = discussion_response.json()

assert discussion["intent"] == "floor_plan_discuss"
assert discussion["assigned_agent"] == "FloorPlanDiscussionAgent"
assert discussion["floor_plan_ready"] is True
assert discussion["floor_plan_missing_fields"] == []
assert discussion["floor_plan_draft"]["rooms"]

floor_plan_draft = discussion["floor_plan_draft"]
{"rooms": floor_plan_draft["rooms"], "ready": discussion["floor_plan_ready"]}


## 4. Continued Discussion With Existing Draft State

This mirrors the dialog sending `temporary_floor_plan_draft` on the next chat turn.

In [ ]:
continued_payload = valid_render_payload("Office 7x9 adjacent with a door")
continued_payload["temporary_floor_plan_draft"] = floor_plan_draft
continued_response = client.post("/agent/orchestrate", json_body=continued_payload)
assert continued_response.status_code == 200, continued_response.text
continued = continued_response.json()

assert continued["intent"] == "floor_plan_discuss"
assert continued["floor_plan_ready"] is True

floor_plan_draft = continued["floor_plan_draft"]
room_names = [room["name"] for room in floor_plan_draft["rooms"]]
assert room_names == ["Living", "Kitchen", "Office"], room_names
room_names


## 5. Plot Through `/agent/orchestrate`

This mirrors clicking `Plot Floor Plan` in SketchUp.

In [ ]:
plot_payload = valid_render_payload("plot the floor plan")
plot_payload["temporary_floor_plan_draft"] = floor_plan_draft
plot_response = client.post("/agent/orchestrate", json_body=plot_payload)
assert plot_response.status_code == 200, plot_response.text
plot = plot_response.json()

assert plot["intent"] == "floor_plan_plot"
assert plot["assigned_agent"] == "FloorPlanToolchain"
assert plot["floor_plan_ready"] is True
assert plot["floor_plan"]["svg_path"].endswith(".svg")
assert plot["floor_plan"]["preview_image_path"].endswith(".png")
assert plot["floor_plan"]["decoration_path"].endswith(".layout.json")
assert [artifact["type"] for artifact in plot["artifacts"]] == ["floor_plan_svg", "floor_plan_png", "floor_plan_decoration_json"]

svg_path = plot["floor_plan"]["svg_path"]
png_path = plot["floor_plan"]["preview_image_path"]
decoration_path = plot["floor_plan"]["decoration_path"]
assert svg_path.endswith(".svg"), svg_path
assert png_path.endswith(".png"), png_path
assert decoration_path.endswith(".layout.json"), decoration_path

plot["floor_plan"]


## 6. Direct `/generate/floor-plan` Endpoint Validation

In [ ]:
direct_response = client.post("/generate/floor-plan", json_body=floor_plan_draft)
assert direct_response.status_code == 200, direct_response.text
direct = direct_response.json()
assert direct["svg_path"].endswith(".svg")
assert direct["preview_image_path"].endswith(".png")
assert direct["decoration_path"].endswith(".layout.json")

invalid_response = client.post(
    "/generate/floor-plan",
    json_body={"rooms": [{"name": "Living", "width": 12, "depth": 14}]},
)
assert invalid_response.status_code == 422
assert "doors/openings" in invalid_response.json()["detail"]

{"direct_status": direct_response.status_code, "invalid_status": invalid_response.status_code}


## 7. Artifact Download Endpoint

In [ ]:
svg_download = client.post("/artifacts/download", json_body={"path": svg_path})
png_download = client.post("/artifacts/download", json_body={"path": png_path})
decoration_download = client.post("/artifacts/download", json_body={"path": decoration_path})
assert svg_download.status_code == 200, svg_download.text
assert png_download.status_code == 200, png_download.text
assert decoration_download.status_code == 200, decoration_download.text

svg_bytes = base64.b64decode(svg_download.json()["content_base64"])
png_bytes = base64.b64decode(png_download.json()["content_base64"])
decoration_bytes = base64.b64decode(decoration_download.json()["content_base64"])
assert svg_bytes.startswith(b"<svg")
assert png_bytes.startswith(b"\x89PNG")
decoration = json.loads(decoration_bytes.decode("utf-8"))
assert decoration["decorated_layout"]["rooms"]
assert decoration["decorated_layout"]["doors"]
assert decoration["decorated_layout"]["furniture"]

local_svg_path = NOTEBOOK_OUTPUT_DIR / Path(svg_path).name
local_png_path = NOTEBOOK_OUTPUT_DIR / Path(png_path).name
local_decoration_path = NOTEBOOK_OUTPUT_DIR / Path(decoration_path).name
local_svg_path.write_bytes(svg_bytes)
local_png_path.write_bytes(png_bytes)
local_decoration_path.write_bytes(decoration_bytes)

{
    "svg_size_bytes": len(svg_bytes),
    "png_size_bytes": len(png_bytes),
    "decoration_size_bytes": len(decoration_bytes),
    "backend_svg_path": svg_path,
    "backend_png_path": png_path,
    "backend_decoration_path": decoration_path,
    "local_svg_path": str(local_svg_path),
    "local_png_path": str(local_png_path),
    "local_decoration_path": str(local_decoration_path),
}


## 8. Visual Preview

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(local_svg_path)))
{
    "authoritative_svg": str(local_svg_path),
    "png_placeholder": str(local_png_path),
}
